# ES Futures Data Preparation

This notebook prepares the CME E-mini S&P 500 futures data used by `OptionsVolatilityAnalysis.ipynb`

It converts the raw Databento `GLBX.MDP3` Statistics download into a clean daily CSV containing settlement price, cleared volume and open interest for the March 2020 (`ESH0`) and June 2020 (`ESM0`) contracts.

In [13]:
# Import packages used to prepare the Databento futures data.
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

## Data scope

The backtest spans 10-20 March 2020 and requires both the March and June ES contracts as the hedge rolls between contracts during the period.

The raw Databento archive remains saved locally in `Data/`, while the processed output is saved as `Data/es_futures_mar2020.csv`.

In [14]:
# Define local data paths and the contracts required by the backtest.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

RAW_ARCHIVES = sorted(DATA_DIR.glob("GLBX-*.zip"))
OUTPUT_PATH = DATA_DIR / "es_futures_mar2020.csv"

TARGET_SYMBOLS = ["ESH0", "ESM0"]

START_DATE = pd.Timestamp("2020-03-10")
END_DATE = pd.Timestamp("2020-03-20")

# Databento statistics: settlement, cleared volume and open interest.
TARGET_STAT_TYPES = [3, 6, 9]

if len(RAW_ARCHIVES) != 1:
    raise RuntimeError(
        f"Expected exactly one Databento GLBX ZIP file in Data/. "
        f"Found {len(RAW_ARCHIVES)}."
    )

RAW_ARCHIVE = RAW_ARCHIVES[0]

print(RAW_ARCHIVE.name)

GLBX-20260902-R8DSQLLHRP.zip


## Load raw Databento data

The Databento batch download contains one Statistics CSV per date. These files are read directly from the ZIP archive and combined into a single DataFrame before filtering.

In [15]:
# Load the daily Statistics CSV files from the Databento archive.
frames = []

with zipfile.ZipFile(RAW_ARCHIVE, "r") as archive:
    statistics_files = sorted(
        filename
        for filename in archive.namelist()
        if filename.endswith(".statistics.csv")
    )

    if not statistics_files:
        raise RuntimeError(
            f"No Databento statistics CSV files found inside {RAW_ARCHIVE.name}."
        )

    for filename in statistics_files:
        with archive.open(filename) as file:
            frames.append(pd.read_csv(file))

stats = pd.concat(frames, ignore_index=True)

print(f"Statistics files loaded: {len(statistics_files)}")
print(f"Raw rows loaded: {len(stats):,}")

Statistics files loaded: 13
Raw rows loaded: 59,512


## Validate the source schema

Before transforming the data, confirm that the Databento fields required for contract identification, statistic type, timestamps and values are present.

This prevents the preparation pipeline from silently running against an unexpected file structure.

In [16]:
# Validate the Databento fields required for the processing pipeline.
required_raw_columns = {
    "symbol",
    "stat_type",
    "ts_ref",
    "ts_recv",
    "price",
    "quantity",
    "stat_flags",
}

missing_raw_columns = required_raw_columns - set(stats.columns)

if missing_raw_columns:
    raise RuntimeError(
        "Missing expected Databento columns: "
        + ", ".join(sorted(missing_raw_columns))
    )

print("Required Databento columns found.")

Required Databento columns found.


## Select the required futures observations

Only the March (`ESH0`) and June (`ESM0`) ES contracts are retained.

The model uses three CME Statistics fields:

- settlement price
- cleared volume
- open interest

Fixing prices are excluded because the June contract does not provide a consistent daily fixing series across the backtest window. 

In [17]:
# Retain the March and June ES contracts and the required CME statistics.
stats = stats[
    stats["symbol"].isin(TARGET_SYMBOLS)
    & stats["stat_type"].isin(TARGET_STAT_TYPES)
].copy()

print(stats.groupby(["symbol", "stat_type"]).size())

symbol  stat_type
ESH0    3            38
        6            19
        9            19
ESM0    3            46
        6            20
        9            20
dtype: int64


## Assign statistics to the correct trading date

CME daily statistics can be published after the trading session for which they relate. Databentos `ts_ref` field identifies the referenced trading session, so it is used to construct the model's daily date index rather than the publication timestamp.

The observations are then restricted to the 10-20 March 2020 backtest window.

In [18]:
# Use ts_ref to assign each CME statistic to its economic trading date.
stats["Date"] = (
    pd.to_datetime(
        stats["ts_ref"],
        utc=True,
        errors="coerce",
    )
    .dt.tz_localize(None)
    .dt.normalize()
)

stats["ts_recv"] = pd.to_datetime(
    stats["ts_recv"],
    utc=True,
    errors="coerce",
)

stats = stats[
    (stats["Date"] >= START_DATE)
    & (stats["Date"] <= END_DATE)
].copy()

print(sorted(stats["Date"].dt.strftime("%Y-%m-%d").unique()))

['2020-03-10', '2020-03-11', '2020-03-12', '2020-03-13', '2020-03-16', '2020-03-17', '2020-03-18', '2020-03-19', '2020-03-20']


## Resolve repeated daily publications

CME can publish more than one observation for the same contract, trading date and statistic.

For settlement prices, final observations are preferred to preliminary values.

Where multiple records otherwise remian, latest published value is retained.

In [19]:
# Prefer final settlements and otherwise retain the latest published statistic.
stats["is_final_settlement"] = (
    (stats["stat_type"] == 3)
    & (
        stats["stat_flags"]
        .fillna(0)
        .astype(int)
        .map(lambda value: bool(value & 1))
    )
)

stats = (
    stats
    .sort_values(
        [
            "Date",
            "symbol",
            "stat_type",
            "is_final_settlement",
            "ts_recv",
        ]
    )
    .groupby(
        ["Date", "symbol", "stat_type"],
        as_index=False,
    )
    .tail(1)
)

## Map Databento statistics to model fields

Each Databento Statistics record contains at least one statistic type.

Settlement prices are stored in the `price` field. Cleared volume and open interest are stored in `quantity`. These records are mapped to common metric names before reshaping the dataset.

In [20]:
# Map Databento statistic types to the fields used by the backtest.
stats["Metric"] = stats["stat_type"].map(
    {
        3: "Settlement",
        6: "Volume",
        9: "OpenInterest",
    }
)

stats["Value"] = np.where(
    stats["stat_type"] == 3,
    stats["price"],
    stats["quantity"],
)

## Reshape to backtest input format

The filtered observations are reshapred to one row per trading date, with separate columns for March and June contracts.

Databento's raw symbols (`ESH0` and `ESM0`) are renamed to existing notebook convention (`ESH20` and `ESM20`). This allows data integration without changing established contract naming in `OptionsVolatilityAnalysis.ipynb`.

In [21]:
# Reshape the observations to one row per trading date.
es_data = stats.pivot(
    index="Date",
    columns=["symbol", "Metric"],
    values="Value",
)

es_data.columns = [
    f"{symbol}_{metric}"
    for symbol, metric in es_data.columns
]

es_data = es_data.rename(
    columns={
        "ESH0_Settlement": "ESH20_Settlement",
        "ESH0_Volume": "ESH20_Volume",
        "ESH0_OpenInterest": "ESH20_OpenInterest",
        "ESM0_Settlement": "ESM20_Settlement",
        "ESM0_Volume": "ESM20_Volume",
        "ESM0_OpenInterest": "ESM20_OpenInterest",
    }
)

required_columns = [
    "ESH20_Settlement",
    "ESH20_Volume",
    "ESH20_OpenInterest",
    "ESM20_Settlement",
    "ESM20_Volume",
    "ESM20_OpenInterest",
]

es_data = (
    es_data
    .reindex(columns=required_columns)
    .sort_index()
    .reset_index()
)

es_data

,Date,ESH20_Settlement,ESH20_Volume,ESH20_OpenInterest,ESM20_Settlement,ESM20_Volume,ESM20_OpenInterest
0,2020-03-10,2865.75,3423140.0,3133378.0,2854.25,195661.0,299064.0
1,2020-03-11,2740.25,2992894.0,3190717.0,2729.00,208617.0,352905.0
2,2020-03-12,2469.00,4366577.0,3112678.0,2456.00,1434076.0,689460.0
3,2020-03-13,2696.00,3423449.0,2691609.0,2684.00,3233288.0,1426136.0
4,2020-03-16,2416.25,2378284.0,2204572.0,2405.25,3539756.0,2156542.0
5,2020-03-17,2495.50,1683796.0,1589082.0,2485.50,3788132.0,2955942.0
6,2020-03-18,2414.00,976536.0,1385001.0,2401.50,3255696.0,3233933.0
7,2020-03-19,2403.25,631228.0,1241170.0,2389.00,3127808.0,3299307.0
8,2020-03-20,2437.98,NaN,NaN,2288.50,3075997.0,3341388.0


## Validate the processed dataset

The processed data is checked for complete trading-date coverage and for missing settlement prices. (both March and June contracts are checked)

Settlement coverage is essential. **Settlement-to-settlement futures are used to calculate the hedge P&L in main backtest.**

Therefore, **missing volume or open-intrerest observations are retained as missing rather than assumed** where they are not required for the active hedge.

In [22]:
# Validate trading-date coverage and the settlement series used by the hedge.
expected_dates = pd.to_datetime(
    [
        "2020-03-10",
        "2020-03-11",
        "2020-03-12",
        "2020-03-13",
        "2020-03-16",
        "2020-03-17",
        "2020-03-18",
        "2020-03-19",
        "2020-03-20",
    ]
)

if not es_data["Date"].equals(
    pd.Series(expected_dates, name="Date")
):
    raise RuntimeError(
        "Processed ES trading dates do not match the expected backtest window."
    )

if es_data[
    ["ESH20_Settlement", "ESM20_Settlement"]
].isna().any().any():
    raise RuntimeError(
        "Missing ES settlement observations were found."
    )

print(es_data.isna().sum())

Date                  0
ESH20_Settlement      0
ESH20_Volume          1
ESH20_OpenInterest    1
ESM20_Settlement      0
ESM20_Volume          0
ESM20_OpenInterest    0
dtype: int64


## Save the model input 

The validated daily dataset is written to `Data/es_futures_mar2020.csv`

The `Data/` directory is exlcuded from version control. This notebook documents **the reproducible transformation process.**

In [ ]:
# Save the validated ES futures dataset used by the main backtest.
es_data.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: c:\devapps\projects\OptionsVolatilityAnalysis\Data\es_futures_mar2020.csv
